In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta


In [ ]:

@sleep_and_retry
@limits(calls=5, period=60)  # Free tier limits
def fetch_data(self, symbol: str, timeframe: str = '1H') -> pd.DataFrame:
    """Fetch historical data using Polygon.io Python client"""
    # Validate timeframe
    if timeframe not in ['1H', '5M']:
        raise ValueError("Invalid timeframe. Use '1H' or '5M'")

    # Calculate date range (last 100 bars)
    end_date = datetime.utcnow()
    if timeframe == '1H':
        start_date = end_date - timedelta(hours=100)
        timespan = 'hour'
        multiplier = 1
    else:  # 5M
        start_date = end_date - timedelta(minutes=500)
        timespan = 'minute'
        multiplier = 5

    try:
        # Get aggregates from Polygon
        aggs: List[Agg] = self.client.get_aggs(
            ticker=symbol,
            multiplier=multiplier,
            timespan=timespan,
            from_=start_date.date().isoformat(),
            to=end_date.date().isoformat(),
            limit=100,
            adjusted=True,
            sort='asc'
        )

        print(f'Fetched {len(aggs)} bars for {symbol} ({timeframe})')

        # Convert to DataFrame
        data = [{
            'Open': agg.open,
            'High': agg.high,
            'Low': agg.low,
            'Close': agg.close,
            'Volume': agg.volume,
            'Timestamp': pd.to_datetime(agg.timestamp, unit='ms')
        } for agg in aggs]

        if not data:
            return pd.DataFrame()

        df = pd.DataFrame(data)
        df.set_index('Timestamp', inplace=True)
        print("df from fetch: ")
        print(df)
        return df.sort_index()

    except polygon.ApiError as e:
        print(f"Polygon API Error: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Unexpected error: {e}")
        return pd.DataFrame()


In [43]:
import alpaca_trade_api as tradeapi
from alpaca.trading.client import TradingClient

import os
from dotenv import load_dotenv

load_dotenv()

api = tradeapi.REST("PKB90UFVEPCKP4HK7Y5A", "Zi5nlLZrtnk4qTYtHwoWES2P7iFkhXiVdamwhn7c", "https://paper-api.alpaca.markets")
trading_client = TradingClient("PKB90UFVEPCKP4HK7Y5A", "Zi5nlLZrtnk4qTYtHwoWES2P7iFkhXiVdamwhn7c", paper=True)

In [19]:
# api.get_account()

In [46]:
api.get_account()

Account({   'account_blocked': False,
    'account_number': 'PA397U0AR1D6',
    'accrued_fees': '0',
    'admin_configurations': {},
    'balance_asof': '2025-04-08',
    'bod_dtbp': '3992813.72',
    'buying_power': '3992813.72',
    'cash': '998203.43',
    'created_at': '2025-02-10T12:10:24.085242Z',
    'crypto_status': 'ACTIVE',
    'crypto_tier': 1,
    'currency': 'USD',
    'daytrade_count': 0,
    'daytrading_buying_power': '3992813.72',
    'effective_buying_power': '3992813.72',
    'equity': '998203.43',
    'id': 'cf841260-a65a-464b-827a-ea6d243f8829',
    'initial_margin': '0',
    'intraday_adjustments': '0',
    'last_equity': '998203.43',
    'last_maintenance_margin': '0',
    'long_market_value': '0',
    'maintenance_margin': '0',
    'multiplier': '4',
    'non_marginable_buying_power': '998203.43',
    'options_approved_level': 3,
    'options_buying_power': '998203.43',
    'options_trading_level': 3,
    'pattern_day_trader': True,
    'pending_reg_taf_fees': '0

In [44]:
def get_account_balance():
    account = api.get_account()
    balance = float(account.buying_power)
    return balance  

In [45]:
get_account_balance()

Account Balance: 3992813.72


3992813.72

In [36]:
import json

In [38]:
SETTINGS_FILE = 'trading_settings.json'

In [37]:
def load_settings():
    default = {
        'tickers': ['AAPL', 'MSFT'],
        'risk_percent': 1,
        'accounts': []
    }
    SETTINGS_FILE = 'trading_settings.json'
    try:
        if os.path.exists(SETTINGS_FILE):
            with open(SETTINGS_FILE) as f:
                loaded = json.load(f)
                # Merge all settings
                default.update(loaded)
                # Validate accounts
                default['accounts'] = [
                    acc for acc in loaded.get('accounts', [])
                    if all(k in acc for k in ('name', 'key', 'secret', 'base_url'))
                ]
    except Exception as e:
        print(f"Error loading settings: {e}")

    return default

In [40]:
load_settings().get('accounts', [])

[{'name': 'Test Account',
  'key': 'PKB90UFVEPCKP4HK7Y5A',
  'secret': 'Zi5nlLZrtnk4qTYtHwoWES2P7iFkhXiVdamwhn7c',
  'base_url': 'https://paper-api.alpaca.markets'}]